# Setting up and administering the visit sequence metadata database

In [1]:
from pathlib import Path
from psycopg2 import sql
from rubin_sim.sim_archive import vseqarchive

## Look at what's in the database

See what schema are already there:

In [2]:
%%bash
psql --host 134.79.23.205 --username rubin --command "\dn" opsim_log

      List of schemas
  Name  |       Owner       
--------+-------------------
 public | pg_database_owner
(1 row)



## Create a test schema

Create an interface to the visit sequence archive metadata database.
Start by setting connection parameters to a user and host with permissions:

In [3]:
metadata_db_kwargs = {
    'database': 'opsim_log',
    'host': '134.79.23.205',
    'user': 'rubin'
}

Make an instance of the interface.
Set the schema to `test`, because (for safety reasons) the interface only allows creation of schema with `test` in the name.

In [4]:
vsarchive = vseqarchive.VisitSequenceArchiveMetadata(
    metadata_db_kwargs,
    metadata_db_schema='test'
)

Run the method that creates an instance of the schema (with name `test`):

In [5]:
vsarchive.create_schema_in_database()

Created test database and schema  test


Check that it was created:

In [6]:
%%bash
psql --host 134.79.23.205 --username rubin --command "\dn" opsim_log

      List of schemas
  Name  |       Owner       
--------+-------------------
 public | pg_database_owner
 test   | rubin
(2 rows)



Rename our newly created `test` schema to the production schema name, so it becomes our production schema:

In [7]:
%%bash
psql --host 134.79.23.205 --username rubin --command "ALTER SCHEMA test RENAME TO vsmd" opsim_log

ALTER SCHEMA


Check that it did what we wanted:

In [8]:
%%bash
psql --host 134.79.23.205 --username rubin --command "\dn" opsim_log

      List of schemas
  Name  |       Owner       
--------+-------------------
 public | pg_database_owner
 vsmd   | rubin
(2 rows)



In [9]:
%%bash
psql --host 134.79.23.205 --username rubin --command "SELECT table_schema, table_name, table_type FROM information_schema.tables WHERE table_schema = 'vsmd';" opsim_log


 table_schema |     table_name      | table_type 
--------------+---------------------+------------
 vsmd         | visitseq            | BASE TABLE
 vsmd         | simulations         | BASE TABLE
 vsmd         | completed           | BASE TABLE
 vsmd         | mixedvisitseq       | BASE TABLE
 vsmd         | tags                | BASE TABLE
 vsmd         | comments            | BASE TABLE
 vsmd         | files               | BASE TABLE
 vsmd         | simulations_extra   | VIEW
 vsmd         | conda_env           | BASE TABLE
 vsmd         | conda_packages      | VIEW
 vsmd         | simulation_packages | VIEW
 vsmd         | nightly_stats       | BASE TABLE
 vsmd         | maf_summary_metrics | BASE TABLE
 vsmd         | maf_metrics         | BASE TABLE
 vsmd         | maf_metric_sets     | BASE TABLE
 vsmd         | maf_summary         | VIEW
 vsmd         | maf_healpix_stats   | BASE TABLE
(17 rows)



I also want an actual schema named `test`, so make it:

In [10]:
vsarchive.create_schema_in_database()

Created test database and schema  test


In [11]:
%%bash
psql --host 134.79.23.205 --username rubin --command "\dn" opsim_log

      List of schemas
  Name  |       Owner       
--------+-------------------
 public | pg_database_owner
 test   | rubin
 vsmd   | rubin
(3 rows)



## Creating users and giving them permissions

Create four users initially:
- `reader` will be a shared account for read-only access.
- `prenight` will be a used by the pre-night simulation process to add pre-night simulations to the database.
- `neilsen` will be used by neilsen for adding data.
- `neilsentest` will be used by neilsen for testing, without write access to the production schema.

Use the `\password` `psql` command to avoid the password being recorded in the `psql` history.

Create groups for the users:

Give the groups permissions: